# 04 · Safety & Hallucination Mitigation

> **Source notes:** `SafetyAndHallucination.md`

A system that is right 95% of the time is dangerous if users can't tell which 5% is wrong. This notebook demonstrates the layered mitigation stack:

- **Hallucination taxonomy** — generating examples of each hallucination type
- **Grounding constraints** — prompt-level mitigation
- **NLI-based claim verification** — post-generation check using a cross-encoder
- **Self-consistency sampling** — flag low-agreement answers
- **Confidence elicitation** — ask the model to rate its own certainty

All LLM calls use **Ollama** (local, no API key needed).

## 0 · Environment Setup

```bash
ollama pull phi3:mini
ollama serve
```

In [ ]:
# TODO: Implement this cell


## 1 · Hallucination Taxonomy — Generating Examples

| Type | Description |
|---|---|
| **Factual hallucination** | Plausible-sounding but wrong fact |
| **Confabulation** | Fabricated citation with correct-looking format |
| **Sycophantic hallucination** | Agreeing with a false premise |
| **Specification overreach** | Adding unrequested content beyond constraints |

We provoke each type to understand what we're defending against.

In [ ]:
# TODO: Implement this cell


## 2 · Prompt-Level Mitigation — Grounding Constraints

The single most effective prompt-level mitigation:

```
"Base your answer ONLY on the provided context.
 If the answer is not present, say: 'I don't have that information.'"
```

We compare an ungrounded prompt vs a grounded prompt with the same out-of-context query.

In [ ]:
# TODO: Implement this cell


## 3 · NLI-Based Claim Verification

After generation, verify each claim in the output is **entailed** by the source context using a Natural Language Inference (NLI) cross-encoder model.

Label mapping:
- **ENTAILMENT** → claim is supported by context ✓
- **NEUTRAL** → claim may or may not be true based on context ⚠
- **CONTRADICTION** → claim is contradicted by context ✗

We use `cross-encoder/nli-deberta-v3-small` (runs locally, ~150 MB).

In [ ]:
# TODO: Implement this cell


## 4 · Self-Consistency Sampling — Detecting Uncertain Answers

Generate **N answers at temperature > 0**, then take the majority vote. Low-agreement answers (no clear majority) are flagged for human review.

This is useful for high-stakes queries where a single-pass answer may be wrong.

In [ ]:
# TODO: Implement this cell


## 5 · Confidence Elicitation

Ask the model to rate its own confidence. Low-confidence answers trigger a fallback (human escalation or "I don't know" response). Note: self-reported confidence is not perfectly calibrated, but it correlates with actual accuracy enough to be useful.

In [ ]:
# TODO: Implement this cell


## Summary — Layered Mitigation Stack

| Layer | Technique | This Notebook |
|---|---|---|
| Prompt-level | Grounding constraint | Section 2 |
| Post-generation | NLI claim verification | Section 3 |
| Post-generation | Self-consistency sampling | Section 4 |
| Post-generation | Confidence elicitation | Section 5 |

Apply all four layers in production for high-stakes outputs. No single technique eliminates hallucination — defence in depth is the only reliable strategy.

**Next:** [EvaluatingAISystems/notebook.ipynb](../EvaluatingAISystems/notebook.ipynb) — systematic evaluation beyond ad-hoc testing.